# detach-clone-snapshot — ex1: snapshot a hidden state across an optimizer step

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `detach-clone-snapshot`. Running the final beacon cell reports progress against the `PyTorch: detach + clone snapshot` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: detach + clone snapshot` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`detach-clone-snapshot`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "detach-clone-snapshot"
DD_SUBTOPIC = "PyTorch: detach + clone snapshot"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## detach + clone — graph-free deep copy refresher

`x.detach()` returns a NEW tensor that shares storage with `x` but is detached from the autograd graph (its `requires_grad` is `False`). `x.clone()` returns a NEW tensor with its OWN storage but stays inside the graph (gradients still flow back through the clone op).

Combine them — `x.detach().clone()` — to take a **graph-free deep copy**. The result has its own storage AND is severed from the graph: perfect for snapshotting hidden states across optimizer steps, logging activations without holding the graph, or stashing a target for a self-distillation step.

**Why both.** `detach()` alone aliases the source — writing to the snapshot mutates the original. `clone()` alone keeps the autograd graph alive — you'd leak memory across training steps. Only the pair gives you the snapshot semantics you actually want.

### Exercise 1 — snapshot a hidden state across an optimizer step

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `.detach().clone()` to snapshot a tensor so the snapshot has its own storage and is severed from the autograd graph.
> Keywords: detach, clone, snapshot, autograd
> ```

**KCs targeted:** `detach-strips-graph`, `clone-copies-storage`

Implement `ex1_snapshot(x)`. Given a leaf tensor `x` with `requires_grad=True`, return a snapshot that:

1. Has its OWN storage (mutating the snapshot must not affect `x`).
2. Has `requires_grad == False` (it is severed from the graph).
3. Has the same values and shape as `x`.

Use `x.detach().clone()`. Order matters for readability — `detach` first severs the graph, `clone` second copies storage. (The opposite order works too, but `detach().clone()` is the idiomatic snapshot.)

Input: any float tensor.
Output: a graph-free deep copy.

In [ ]:
def ex1_snapshot(x: Tensor) -> Tensor:
    return x.detach().clone()


<details><summary>Solution</summary>

```python
def ex1_snapshot(x: Tensor) -> Tensor:
    return x.detach().clone()
```

**Why `.detach()` first.** Reading `detach().clone()` left-to-right is 'sever the graph, then copy storage' — which matches what you want semantically. `clone().detach()` is identical at runtime but reads backwards.

**Why not `.data`.** `x.data` looks tempting but is a footgun: it aliases storage AND silently violates autograd's view tracking. Prefer `detach()` in modern code.

**Common use cases.** Logging hidden states without keeping the graph alive; building a teacher target in self-distillation; stashing the last-step parameters for a EMA / Polyak average.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()